# Agent Network Data Cleanup: Audit, Clean, Merge, Summarise
**Engagement 2.** The legacy transaction extract is unreliable, so I audit it, clean it with `clean_agent_data()` (`cleaning_pipeline.py`), merge it with the agent register and summarise it.

## Task 10: Data-quality audit
**Approach:** read every column as text with `keep_default_na=False` so blanks and the text `N/A` can be counted separately (pandas turns `N/A` into missing by default).

In [1]:
import os, subprocess, sys
import pandas as pd
from cleaning_pipeline import clean_agent_data
pd.set_option("display.float_format", "{:,.0f}".format); pd.set_option("display.width", 110); pd.set_option("display.max_columns", 20)
if not os.path.exists("agent_transactions.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)
raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)
agents = pd.read_csv("agents.csv")
audit = pd.DataFrame([
    ("Duplicate transactions (repeated txn_id)", "Uniqueness", "txn_id", raw["txn_id"].duplicated().sum()),
    ("txn_type spelled inconsistently", "Consistency", "txn_type", (~raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("Dates written with '/' not '-'", "Consistency", "txn_date", raw["txn_date"].str.contains("/").sum()),
    ("Dates and amounts stored as text", "Validity", "txn_date, amount", len(raw)),
    ("Blank amount", "Completeness", "amount", (raw["amount"] == "").sum()),
    ("Text 'N/A' in amount", "Validity", "amount", (raw["amount"] == "N/A").sum()),
    ("Amount of exactly 250000 (repeated)", "Accuracy", "amount", (raw["amount"] == "250000").sum()),
    ("agent_id missing from register (AG099)", "Consistency", "agent_id", (~raw["agent_id"].isin(agents["agent_id"])).sum()),
], columns=["issue", "dimension", "column", "rows_affected"])
print("Raw extract:", raw.shape, "| agents:", agents.shape)
audit

Raw extract: (612, 5) | agents: (20, 3)


                                      issue     dimension            column  rows_affected
0  Duplicate transactions (repeated txn_id)    Uniqueness            txn_id             12
1           txn_type spelled inconsistently   Consistency          txn_type            384
2            Dates written with '/' not '-'   Consistency          txn_date              5
3          Dates and amounts stored as text      Validity  txn_date, amount            612
4                              Blank amount  Completeness            amount             15
5                      Text 'N/A' in amount      Validity            amount              7
6       Amount of exactly 250000 (repeated)      Accuracy            amount              5
7    agent_id missing from register (AG099)   Consistency          agent_id              8

**Answer.** **8 issue types** in the 612-row extract. Some counts exceed the injected ones because the 12 duplicates also copied 2 `N/A` amounts (7 raw vs 5 unique). A timeliness check found no transaction dated before its agent's onboarding.

## Tasks 11-15: Cleaning pipeline
**Approach:** every rule lives in `clean_agent_data(txns, agents)`, which returns the clean DataFrame and a log, so the same function can clean next month's extract.

In [2]:
clean_df, log = clean_agent_data(pd.read_csv("agent_transactions.csv"), agents)
pd.DataFrame(log)[["step", "issue", "dimension", "rows_affected"]]

   step                                              issue     dimension  rows_affected
0     1               Duplicate transactions (same txn_id)    Uniqueness             12
1     2                 txn_type has mixed case and spaces   Consistency            376
2     3              Dates written with '/' instead of '-'   Consistency              5
3     4         Missing amounts (blank, 'N/A' read as NaN)  Completeness             20
4     5           Non-numeric text or non-positive amounts      Validity              0
5     6              Suspiciously large amounts (IQR rule)      Accuracy             40
6     7                 Missing or invalid amounts imputed  Completeness             20
7     8       Transactions from agents not in the register   Consistency              7
8     9  Transactions dated before the agent was onboarded    Timeliness              0

In [3]:
obs = clean_df.loc[~clean_df["amount_imputed"], "amount"]
q1, q3 = obs.quantile([0.25, 0.75]); fence = q3 + 1.5 * (q3 - q1)
imp, big, tot = clean_df[clean_df["amount_imputed"]], clean_df[clean_df["amount"] == 250000], clean_df["amount"].sum()
print(f"Rows: raw {len(raw)} -> clean {len(clean_df)} | txn_id unique: {clean_df['txn_id'].is_unique}")
print("txn_type:", clean_df["txn_type"].value_counts().to_dict(), "| dates parsed:", int(clean_df["txn_date"].notna().sum()))
print(f"Imputed: {len(imp)} rows, KES {imp['amount'].sum():,.0f} ({imp['amount'].sum() / tot:.1%} of value) | observed mean {obs.mean():,.0f}, median {obs.median():,.0f}")
print(f"IQR fence KES {fence:,.0f}: {int(clean_df['is_outlier'].sum())} flagged | 250,000 rows: {len(big)}, "
      f"KES {big['amount'].sum():,.0f} ({big['amount'].sum() / tot:.1%} of value, {250000 / obs.median():.0f}x median)")

Rows: raw 612 -> clean 600 | txn_id unique: True
txn_type: {'Deposit': 373, 'Withdrawal': 227} | dates parsed: 600
Imputed: 20 rows, KES 33,843 (1.3% of value) | observed mean 4,574, median 1,776
IQR fence KES 6,084: 40 flagged | 250,000 rows: 5, KES 1,250,000 (46.5% of value, 141x median)


**Answers (612 rows become 600; nothing else is dropped).**
- **11 Uniqueness:** **12 duplicates removed** (11 exact copies). `TX00405` was copied with agent `AG099`, so I kept the copy with the registered agent `AG014`.
- **12 Consistency:** five spellings mapped to exactly two values (373 Deposit, 227 Withdrawal, 0 unmapped, `isna()` checked).
- **13 Validity:** all 600 dates parse, including the 5 written with `/`; amounts are numeric.
- **14 Completeness:** 20 amounts are missing (15 blank, 5 `N/A`). Dropping them would delete real transactions and the mean (4,574) is skewed by outliers, so I filled each with the **median by type** (KES 1,717 Deposit, 1,646 Withdrawal) and set `amount_imputed`. They add only **1.3%** of value (KES 33,843).
- **15 Outliers:** the IQR fence of KES 6,084 flags 40 amounts. 35 are ordinary large values (max KES 28,847). **5 are exactly KES 250,000**: 141x the median, KES 1,250,000, **46.5% of all value**, across five agents, which looks like a system default or test value. **Recommendation:** keep but flag them, report totals with and without them, and verify with the agents before use.

## Task 16: Merge and post-merge validation
**Approach:** a left join on `agent_id` (validated many-to-one) keeps every transaction. Unknown agents get county `Unknown` instead of being dropped or guessed.

In [4]:
unk = clean_df[clean_df["unknown_agent"]]
print("Rows before / after merge:", raw["txn_id"].nunique(), "/", len(clean_df), "| register agents:", len(agents))
print("Unknown-agent transactions:", len(unk), unk["agent_id"].unique().tolist(), f"| KES {unk['amount'].sum():,.0f}")
print("Independent check (not in register):", int((~clean_df["agent_id"].isin(agents["agent_id"])).sum()))

Rows before / after merge: 600 / 600 | register agents: 20
Unknown-agent transactions: 7 ['AG099'] | KES 27,156
Independent check (not in register): 7


**Answer.** All **600 transactions survive** the merge (600 before and after). **7 transactions** (KES 27,156, 1.0% of value) reference unknown agent `AG099` (8 in the raw extract; one was a removed duplicate). They stay labelled `Unknown`; the governance team should check whether `AG099` is a missing registration or a keying error.

## Task 17: County summary and weekly pivot
**(a)** County by transaction type. **(b)** Weekly value (week starting Monday) by county; the first and last weeks are partial.

In [5]:
county = (clean_df.groupby(["county", "txn_type"])["amount"]
          .agg(total_value="sum", transactions="count", avg_value="mean").reset_index())
print(county.to_string(index=False))
ranked = pd.DataFrame({"all": clean_df.groupby("county")["amount"].sum(),
                       "excl_250k_imputed": clean_df[~clean_df["amount_imputed"] & (clean_df["amount"] != 250000)]
                       .groupby("county")["amount"].sum()}).sort_values("all", ascending=False)
print(ranked.to_string())
weekly = clean_df.assign(week=clean_df["txn_date"].dt.to_period("W").dt.start_time.dt.date)
weekly.pivot_table(index="week", columns="county", values="amount", aggfunc="sum", fill_value=0,
                   margins=True, margins_name="Total")

 county   txn_type  total_value  transactions  avg_value
 Kiambu    Deposit      334,737            43      7,785
 Kiambu Withdrawal       76,920            27      2,849
 Kisumu    Deposit      436,242            74      5,895
 Kisumu Withdrawal       76,244            37      2,061
Mombasa    Deposit      148,748            72      2,066
Mombasa Withdrawal      368,592            61      6,042
Nairobi    Deposit      543,352           118      4,605
Nairobi Withdrawal      161,514            62      2,605
 Nakuru    Deposit      184,771            60      3,080
 Nakuru Withdrawal      328,464            39      8,422
Unknown    Deposit       25,279             6      4,213
Unknown Withdrawal        1,877             1      1,877
            all  excl_250k_imputed
county                            
Nairobi 704,866            441,414
Mombasa 517,340            260,472
Nakuru  513,235            258,155
Kisumu  512,486            255,689
Kiambu  411,657            160,011
Unknown  27,15

county      Kiambu  Kisumu  Mombasa  Nairobi  Nakuru  Unknown     Total
week                                                                   
2026-02-23       0   1,893    6,759    7,046   2,684        0    18,382
2026-03-02  18,317 287,933  285,610   68,967  56,574        0   717,401
2026-03-09 267,641  23,186   30,418   52,801  28,730        0   402,776
2026-03-16  22,054  27,935   25,085   43,048  39,795    1,639   159,556
2026-03-23  11,504  27,077   17,050   41,314  18,868        0   115,813
2026-03-30  22,609  41,077   38,344  282,931  18,145    8,507   411,613
2026-04-06  15,030  15,341   47,033   56,727 276,609    9,824   420,564
2026-04-13  12,265  29,297   26,730   56,302  26,093        0   150,687
2026-04-20  24,507  37,439   16,687   64,926  30,862    1,877   176,298
2026-04-27  17,730  21,308   23,624   30,804  14,875    5,309   113,650
Total      411,657 512,486  517,340  704,866 513,235   27,156 2,686,740

**Answer.** **Nairobi leads: KES 704,866 (26.2% of value, 180 transactions)**, then Mombasa (517,340), Nakuru (513,235), Kisumu (512,486) and Kiambu (411,657). Deposits are 62.3% of value (KES 1,673,129). The ranking is fragile: the five 250,000 amounts are 46.5% of value, and without them and the imputed amounts Nairobi's lead grows (KES 441,414 vs Mombasa 260,472). In the pivot, the four weeks containing a 250,000 payment run KES 402,776 to 717,401, against KES 113,650 to 176,298 for other full weeks.

## Export and final checks

In [6]:
os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
c = pd.read_csv("outputs/agent_transactions_clean.csv")
assert c["txn_id"].is_unique and set(c["txn_type"]) == {"Deposit", "Withdrawal"}
assert c["amount"].notna().all() and c["txn_date"].str.match(r"^\d{4}-\d{2}-\d{2}$").all()
assert {"amount_imputed", "is_outlier", "county"} <= set(c.columns) and (len(raw), len(c)) == (612, 600)
assert int(c["amount_imputed"].sum()) == 20 and int(c["is_outlier"].sum()) == 40 and int(c["unknown_agent"].sum()) == 7
assert int((c["amount"] == 250000).sum()) == 5 and round(c["amount"].sum()) == 2686740
print("All checks passed. Saved", c.shape, "to outputs/agent_transactions_clean.csv")

All checks passed. Saved (600, 9) to outputs/agent_transactions_clean.csv


## Data Quality Report
612 rows (1 Mar to 29 Apr 2026) and 20 agents became **600 unique transactions**; every treatment is logged by `clean_agent_data()`.

| Dimension | Finding | Treatment |
|---|---|---|
| **Completeness** | 20 amounts missing (15 blank, 5 `N/A`) | Median by type (KES 1,717 / 1,646), flagged `amount_imputed`; 1.3% of value |
| **Uniqueness** | 12 repeated `txn_id` rows (1 with a wrong agent) | Removed; kept the registered-agent copy |
| **Validity** | Text amounts and dates; `N/A`; 5 dates with `/` | `to_numeric` and explicit date format; all 600 valid |
| **Consistency** | 5 spellings of 2 types (376 rows); 7 rows from unregistered `AG099` | Mapped to Deposit/Withdrawal; county `Unknown`, flagged `unknown_agent` |
| **Accuracy** | 5 amounts of exactly KES 250,000 (46.5% of value, 141x median); 35 other outliers look genuine | Flagged `is_outlier`, not deleted; verify with agents |
| **Timeliness** | 60 days, no gaps, none before onboarding | No action |

**Limitations:** imputed values are estimates and the 250,000 amounts are unverified, so report totals with and without them.

**Privacy (Kenya Data Protection Act 2019).** This extract has no customer identifiers, but customer **phone numbers** would be personal data, and with `agent_id`, `txn_date` and `amount` they would reveal financial behaviour and location patterns; agent identities are personal data too. Under the Act's data-minimisation and purpose-limitation principles I would not collect phone numbers unless needed, replace them with a salted-hash pseudonym before analysis, keep the key separately under restricted access, publish only county or week aggregates, set a retention limit, and record the lawful basis and a data protection impact assessment before processing.